In [1]:
import unicodedata
import numpy as np
import pandas as pd
from keras.src.utils.module_utils import tensorflow
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Input,LSTM,Embedding,MultiHeadAttention,LayerNormalization
import os
from tensorflow.keras.callbacks import EarlyStopping
import datasets
from datasets import Dataset,DatasetDict
import tensorflow as tf
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import ctypes

C:\Users\LOQ\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)

-2147483648

In [3]:
with open ("D:/project_deeplearning/TEP.en-fa.en",encoding="utf-8") as f:
    en_s=f.read().splitlines()

with open ("D:/project_deeplearning/TEP.en-fa.fa",encoding="utf-8") as f:
    fa_s=f.read().splitlines()


assert len(en_s)==len(fa_s)

df=pd.DataFrame(
    {
        "en":en_s,
        "fa":fa_s
    }
)

df=df.sample(n=30000,random_state=42)
df=df.reset_index(drop=True)

en=df["en"]
fa=df["fa"]
data=pd.DataFrame({"train":Dataset.from_pandas(df)})
print(data["train"][0])

{'en': 'stop her . somebody stop her reading .', 'fa': 'متوقفش كنيد يک نفر نگذاره اون ادامه بده .'}


In [4]:
train=data["train"]
train[:10]

0    {'en': 'stop her . somebody stop her reading ....
1        {'en': 'tetrastichous .', 'fa': 'چهاربيتي .'}
2    {'en': 'thats her stage name . i just said tha...
3    {'en': 'i wanna go . what .', 'fa': 'ميخوام بر...
4    {'en': 'im going to see the dragon warrior .',...
5    {'en': 'just think that i've received them and...
6    {'en': 'and the food was no different from a p...
7    {'en': 'are a couple jerkoffs .', 'fa': '2تا ا...
8    {'en': 'i think i should probably just stay wi...
9                {'en': 'vail .', 'fa': 'بکارخوردن .'}
Name: train, dtype: object

In [5]:
def unicode_to_asci(s):
    return "".join(c for c in unicodedata.normalize("NFC",s) if unicodedata.category(c)!='MN' )

In [6]:
len(data)

30000

In [7]:
def preprossing(w):
    w=unicode_to_asci(w.lower().strip())
    w=re.sub(r"([.!?])",r"\1",w)
    w=re.sub(r'([""])+',"",w)
    w=w.rstrip().strip()
    w = "<start> " + w + " <end>"
    return w

In [8]:
en_sen="Im very happy."
preprossing(en_sen)

'<start> im very happy. <end>'

In [9]:
fa_sen="درود بر تو."
preprossing(fa_sen)

'<start> درود بر تو. <end>'

In [10]:
df["en"]=df["en"].apply(preprossing)
df["fa"]=df["fa"].apply(preprossing)

In [11]:
vocab_size=15000
max_length=50
batch_size=16

token_en=Tokenizer(num_words=vocab_size,filters="",oov_token="<unk>")
token_fa=Tokenizer(num_words=vocab_size,filters="",oov_token="<unk>")



token_en.fit_on_texts(df["en"])
token_fa.fit_on_texts(df["fa"])

en_seq=token_en.texts_to_sequences(df["en"])
fa_seq=token_fa.texts_to_sequences(df["fa"])

en_seq=pad_sequences(en_seq , maxlen=max_length,padding="post")
fa_seq=pad_sequences(fa_seq,maxlen=max_length,padding="post")

decoder_inputs_array=fa_seq[:,:-1]
decoder_targets_array=fa_seq[:,1:]




In [12]:
latent_dim=256

encoder_inputs=Input(shape=(max_length,),name="encoder_inputs")
encoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True)(encoder_inputs)
encoder_output,state_h,state_c=LSTM(latent_dim,return_state=True,return_sequences=True,dropout=0.2)(encoder_embedding)




decoder_inputs=Input(shape=(None,),name="decoder_inputs")
decoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True,name="decoder_embedding")
decoder_embedd=decoder_embedding(decoder_inputs)
decoder_lstm=LSTM(latent_dim,return_sequences=True,return_state=True,name="decoder_lstm",dropout=0.2)
decoder_outputs,state_h_dec,state_c_dec=decoder_lstm(decoder_embedd,initial_state=[state_h,state_c])

attention_layers=MultiHeadAttention(num_heads=4,key_dim=64,name='multi_head_attention')
attn=attention_layers(
    query=decoder_outputs,
    key=encoder_output,
    value=encoder_output
)

x=decoder_outputs+attn
nor=LayerNormalization()
x=nor(x)

In [13]:
decoder_dense=Dense(vocab_size,activation="softmax",kernel_regularizer=tf.keras.regularizers.l2(1e-3))
decoder_outputs=decoder_dense(x)

In [14]:
model=tf.keras.Model([encoder_inputs,decoder_inputs],decoder_outputs)

In [15]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 256)   │  3,840,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 50)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 256) │  3,840,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 50, 256), │    525,312 │ embedding[0][0],  │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │    525,312 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 256) │    263,168 │ lstm[0][0],       │
│ (MultiHeadAttentio… │                   │            │ decoder_lstm[0][… │
│                     │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None, 256) │          0 │ decoder_lstm[0][… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, None, 256) │        512 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  3,855,000 │ layer_normalizat… │
│                     │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,849,304 (49.02 MB)

 Trainable params: 12,849,304 (49.02 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
splt=int(len(en_seq)*0.9)
train_ds=tf.data.Dataset.from_tensor_slices(((en_seq[:splt],decoder_inputs_array[:splt]),decoder_targets_array[:splt])).shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds=tf.data.Dataset.from_tensor_slices(((en_seq[splt:],decoder_inputs_array[splt:]),decoder_targets_array[splt:])).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [17]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),metrics=["accuracy"],loss="sparse_categorical_crossentropy")

In [18]:
history=model.fit(train_ds,epochs=100,validation_data=val_ds,verbose=2)

Epoch 1/100
1688/1688 - 674s - 400ms/step - accuracy: 0.8709 - loss: 1.1419 - val_accuracy: 0.8784 - val_loss: 0.9549
Epoch 2/100
1688/1688 - 799s - 473ms/step - accuracy: 0.8800 - loss: 0.9490 - val_accuracy: 0.8820 - val_loss: 0.9092
Epoch 3/100
1688/1688 - 804s - 476ms/step - accuracy: 0.8852 - loss: 0.8942 - val_accuracy: 0.8846 - val_loss: 0.8761
Epoch 4/100
1688/1688 - 673s - 399ms/step - accuracy: 0.8895 - loss: 0.8495 - val_accuracy: 0.8861 - val_loss: 0.8589
Epoch 5/100
1688/1688 - 248s - 147ms/step - accuracy: 0.8934 - loss: 0.8109 - val_accuracy: 0.8868 - val_loss: 0.8513
Epoch 6/100
1688/1688 - 248s - 147ms/step - accuracy: 0.8973 - loss: 0.7763 - val_accuracy: 0.8878 - val_loss: 0.8510
Epoch 7/100
1688/1688 - 250s - 148ms/step - accuracy: 0.9012 - loss: 0.7452 - val_accuracy: 0.8884 - val_loss: 0.8488
Epoch 8/100
1688/1688 - 249s - 148ms/step - accuracy: 0.9047 - loss: 0.7156 - val_accuracy: 0.8879 - val_loss: 0.8559
Epoch 9/100
1688/1688 - 249s - 148ms/step - accuracy: 0.

KeyboardInterrupt: 

In [ ]:
reverse_fa = {v: k for k, v in token_fa.word_index.items()}

In [ ]:
import pickle
model.save('Translator.keras')

In [ ]:
from tensorflow.keras.models import load_model
import pickle
model=load_model("Translator.keras")
pickle.dump(token_fa,open("token_fa.pkl","wb"))
pickle.dump(token_en,open("token_en.pkl","wb"))
pickle.dump(reverse_fa,open("reverse_fa.pkl","wb"))

In [ ]:
import pickle

token_fa=pickle.load(open("token_fa.pkl","rb"))
token_en=pickle.load(open("token_en.pkl","rb"))
reverse_fa=pickle.load(open("reverse_fa.pkl","rb"))


In [ ]:
encoder_model=tf.keras.Model(encoder_inputs,[encoder_output,state_h,state_c])

In [ ]:
decoder_state_input_h=Input(shape=(latent_dim,))
decoder_state_input_c=Input(shape=(latent_dim,))
enc_out_input=Input(shape=(max_length,latent_dim))
decoder_states_inputs=[decoder_state_input_h, decoder_state_input_c]

decoder_emb2=decoder_embedding(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_emb2,
    initial_state=decoder_states_inputs
)

attn2=attention_layers(query=decoder_outputs2,key=enc_out_input,value=enc_out_input)
x2=decoder_outputs2+attn2
x2=LayerNormalization()(x2)
decoder_outputs2=decoder_dense(x2)

decoder_model=tf.keras.Model(
    [decoder_inputs,decoder_state_input_h,decoder_state_input_c,enc_out_input],
    [decoder_outputs2, state_h2, state_c2]
)



In [ ]:
encoder_model.save('encoder_model.keras')
decoder_model.save('decoder_model.keras')

In [ ]:
def translate(sentence):

    sentence=preprossing(sentence)

    seq=token_en.texts_to_sequences([sentence])
    seq=pad_sequences(seq, maxlen=max_length, padding="post")

    enc_out,h,c=encoder_model.predict(seq)
    target_seq=np.array([[token_fa.word_index["<start>"]]])

    stop=False
    decoded= ""

    while not stop:

        output_tokens, h, c=decoder_model.predict([target_seq,h,c,enc_out])
        sampled_token_index=np.argmax(output_tokens[0, -1, :])
        sampled_word=reverse_fa.get(sampled_token_index, "")

        if sampled_word== "<end>" or len(decoded.split())>max_length:
            stop=True
        else:
            decoded+= " " +sampled_word

        target_seq=np.array([[sampled_token_index]])
        states=[h, c]

    return decoded


In [ ]:
print(translate("I love you"))


In [ ]:
print(translate('you can'))

In [ ]:
import sacrebleu

In [ ]:
print(tf.config.list_physical_devices('GPU'))

In [ ]:
print(token_fa.index_word[1])
print(token_fa.index_word[2])
print(token_fa.index_word[3])
print(token_fa.index_word[4])
print(token_fa.index_word[5])

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))